In [13]:
import os
import numpy as np
import pandas as pd
import vsp
from pyDOE2 import lhs  # LHS를 위한 패키지, 없다면 pip install pyDOE2

# ============================
# 설정 부분
# ============================
TEMPLATE_VSP3 = "example_output.vsp3"  # 템플릿 파일명
OUTPUT_DIR = "generated_vsp_files"    # 생성 파일 저장 폴더
N_SAMPLES = 10                        # 생성할 샘플 수

# 설계변수 범위 설정
SPAN_RANGE = (15.0, 25.0)   # (최소, 최대)
AREA_RANGE = (30.0, 50.0)   # (최소, 최대)

# ============================
# 생성 폴더 없으면 만들기
# ============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================
# LHS 샘플링
# ============================
np.random.seed(42)  # 재현성 보장
lhs_samples = lhs(2, samples=N_SAMPLES)

# 스케일링
spans = SPAN_RANGE[0] + (SPAN_RANGE[1] - SPAN_RANGE[0]) * lhs_samples[:, 0]
areas = AREA_RANGE[0] + (AREA_RANGE[1] - AREA_RANGE[0]) * lhs_samples[:, 1]

# ============================
# 각 샘플에 대해 VSP 파일 생성
# ============================
for idx, (span, area) in enumerate(zip(spans, areas)):
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(TEMPLATE_VSP3)

    wing_id = vsp.FindGeom("WingGeom", 0)
    
    # TotalSpan, TotalArea 수정
    vsp.SetParmVal(wing_id, "TotalSpan", "WingGeom", span)
    vsp.SetParmVal(wing_id, "TotalArea", "WingGeom", area)
    vsp.Update()

    # 파일 저장
    save_path = os.path.join(OUTPUT_DIR, f"sample_{idx:03d}.vsp3")
    vsp.WriteVSPFile(save_path)
    print(f"✅ 샘플 {idx} 저장 완료: Span={span:.2f}, Area={area:.2f}")

print("\n🎯 모든 샘플링된 VSP 파일 생성 완료!")

# ============================
# 결과 CSV로도 저장 (추적용)
# ============================
sampling_data = pd.DataFrame({
    "Index": range(N_SAMPLES),
    "TotalSpan": spans,
    "TotalArea": areas
})

sampling_data.to_csv(os.path.join(OUTPUT_DIR, "sampling_data.csv"), index=False)
print("📄 샘플링 데이터 CSV 저장 완료!")


✅ 샘플 0 저장 완료: Span=21.04, Area=48.33
✅ 샘플 1 저장 완료: Span=18.10, Area=44.79
✅ 샘플 2 저장 완료: Span=15.02, Area=39.59
✅ 샘플 3 저장 완료: Span=23.92, Area=43.57
✅ 샘플 4 저장 완료: Span=20.04, Area=30.29
✅ 샘플 5 저장 완료: Span=16.33, Area=34.69
✅ 샘플 6 저장 완료: Span=19.37, Area=40.81
✅ 샘플 7 저장 완료: Span=24.42, Area=46.73
✅ 샘플 8 저장 완료: Span=22.02, Area=37.54
✅ 샘플 9 저장 완료: Span=17.71, Area=32.99

🎯 모든 샘플링된 VSP 파일 생성 완료!
📄 샘플링 데이터 CSV 저장 완료!
